# MAL PRATICAL EVALUATION 2025

- Authors : Zakaria El Mrani & Malek Frikha
- Professeur : M. Mougeot

# Mise en Contexte

Nous disposons d’un jeu de données `celldata.csv` décrivant 8 000 clients d’un opérateur mobile. 
Chaque observation contient différentes informations sur les clients (données socio-démographiques 
et contractuelles) ainsi qu’une variable cible indiquant si le client a quitté l’opérateur 
(*churn*).

L’objectif de ce travail est :
1. de construire un modèle de prédiction du churn ;
2. de comparer plusieurs modèles de classification vus en cours selon différents critères 
   (performance, complexité, interprétabilité, etc.) ;
3. d’analyser la *fairness* des prédictions en considérant la caractéristique de genre (`Gender`) 
   comme attribut sensible.


# Exploration du Dataset

In [9]:
import pandas as pd
df = pd.read_csv('celldata.csv')
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Salary,Churn
0,632,Germany,Female,50,5,107959.39,1,1,1,6985,1
1,649,France,Female,42,7,0.00,2,0,1,22974,0
2,595,France,Male,29,6,150685.79,1,1,0,87771,0
3,653,Spain,Male,35,6,116662.96,2,1,1,23864,0
4,559,Spain,Female,40,7,144470.77,1,1,1,18918,0


Nous possédons dans notre jeu de données 8 000 entrées correspondant à 8 000 clients, 
ainsi que 10 variables explicatives et une variable cible `Churn` :

- **CreditScore** : score de crédit du client, représentant la probabilité qu’il paie ses factures
  à temps. Plus le `CreditScore` est élevé, plus le client est considéré comme fiable.
- **Geography** : pays ou région de résidence du client (par exemple : France, Espagne, Allemagne).
- **Gender** : genre du client (`Male` / `Female`).
- **Age** : âge du client (en années).
- **Tenure** : ancienneté du client chez l’opérateur (en années).
- **Balance** : solde du compte du client auprès de l’opérateur (montant détenu).
- **NumOfProducts** : nombre de produits ou services souscrits par le client.
- **HasCrCard** : indicateur binaire valant 1 si le client possède une carte de crédit, 0 sinon.
- **IsActiveMember** : indicateur binaire valant 1 si le client est considéré comme “actif”, 0 sinon.
- **Salary** : estimation du revenu annuel du client.

- **Churn** : variable cible, valant 1 si le client a quitté l’opérateur (churn), 0 sinon.


In [20]:
df.describe(include="all")

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Salary,Churn
count,8000.000000,8000,8000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000
unique,NaN,3,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,France,Male,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,4038,4373,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,650.805000,NaN,NaN,38.871625,5.013000,76011.635494,1.533625,0.704750,0.515750,100211.396750,0.201125
std,96.721648,NaN,NaN,10.446006,2.897565,62402.105212,0.583448,0.456184,0.499783,57165.688096,0.400866
min,350.000000,NaN,NaN,18.000000,0.000000,0.000000,1.000000,0.000000,0.000000,12.000000,0.000000
25%,584.000000,NaN,NaN,32.000000,2.000000,0.000000,1.000000,0.000000,0.000000,51464.500000,0.000000
50%,652.000000,NaN,NaN,37.000000,5.000000,96846.565000,1.000000,1.000000,1.000000,100583.000000,0.000000
75%,718.000000,NaN,NaN,44.000000,7.000000,127593.625000,2.000000,1.000000,1.000000,149068.000000,0.000000


### Overview

- Le jeu de données ne contient aucune valeurs manquantes sur les 10 variables explicatives.
- Les variables catégorielles `Geography` et `Gender` présentent respectivement 3 et 2 modalités, avec une majorité de clients situés en France (environ 50 %) et une légère sur-représentation des hommes (~55 %).
- La variable cible `Churn` est déséquilibrée : environ 20 % des clients ont quitté l’opérateur, contre 80 % restés, ce qui doit être pris en compte dans l’évaluation des modèles.
- Les variables numériques présentent des ordres de grandeur variés : l’âge moyen est d’environ 39 ans, avec une distribution concentrée entre 30 et 45 ans, l’ancienneté (`Tenure`) est centrée autour de 5 ans, et le solde (`Balance`) montre une distribution très asymétrique avec au moins 25 % des clients ayant un solde nul. Enfin, environ 70 % des clients possèdent une carte de crédit et un peu plus de la moitié sont considérés comme membres actifs.

## PreProcessing

Afin de pouvoir utiliser nos modèles de manière efficace, nous devons encoder les variables
catégorielles. Comme ces variables représentent des catégories sans ordre naturel (par exemple
`Geography`, `Gender`), nous utilisons un encodage *one-hot* via `OneHotEncoder`.

Nous découpons ensuite le jeu de données en un ensemble d’entraînement et un ensemble de test,
afin d’ajuster les modèles sur les données d’entraînement et d’évaluer leurs performances sur
des données jamais vues.


In [25]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = df.drop(columns=["Churn"])
y = df["Churn"]

cat_cols = ["Geography", "Gender"]
num_cols = [c for c in X.columns if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first"), cat_cols),
    ]
)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)